# D-SAT — Phase 2: Fine-tuning Gemma 3 on Graph-to-Graph Translation

**Objective:** Fine-tune Gemma 3 (4B instruct) to predict the next scene graph G_t+1 given a current scene graph G_t and an action description.

**Input:** `triplets.jsonl` from Phase 1  
**Output:** A LoRA-adapted Gemma 3 checkpoint acting as the causal transition model

**Pipeline stages:**
```
1. Environment setup
2. Load and inspect triplets
3. Format triplets as text prompts
4. Load Gemma 3 and attach LoRA adapters
5. Train
6. Evaluate on held-out examples (graph edit distance + exact match)
7. Inference demo
8. Save checkpoint
```

> **Runtime:** A100 GPU (required for Gemma 3 4B with 4-bit quantization).  
> **Prerequisites:** Upload `triplets.jsonl` from Phase 1. Accept the Gemma 3 license at `huggingface.co/google/gemma-3-4b-it` and add your token as `HF_TOKEN` in Colab Secrets.

---
## Stage 1 — Environment setup

In [ ]:
%%capture
!pip install transformers peft accelerate bitsandbytes datasets networkx sentencepiece protobuf
!pip install torch --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import os, json, re, math, time
from pathlib import Path
from typing import Optional

import torch
import networkx as nx
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from torch.utils.data import Dataset

ROOT           = Path("/content/dsat_phase2")
ROOT.mkdir(exist_ok=True)
TRIPLETS_FILE  = Path("/content/triplets.jsonl")
CHECKPOINT_DIR = ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("HuggingFace token: ")

MODEL_ID = "google/gemma-3-4b-it"

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected.")
print("Setup complete.")

---
## Stage 2 — Load and inspect triplets

An 80/20 train/val split is applied. At pilot scale (26 triplets) this yields 20 train and 6 val examples — sufficient to verify the pipeline runs end-to-end. Scale to 90/10 when Phase 1 is run at full scale (3000+ triplets).

In [ ]:
triplets = []
with open(TRIPLETS_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            triplets.append(json.loads(line))

print(f"Loaded {len(triplets)} triplets")
print(f"Keys: {list(triplets[0].keys())}")
print(f"\nSample action : {triplets[0]['action_text']}")
print(f"G_t objects   : {[o['class'] for o in triplets[0]['g_t']['objects']]}")
print(f"G_t+1 objects : {[o['class'] for o in triplets[0]['g_t1']['objects']]}")

In [ ]:
import random
random.seed(42)
random.shuffle(triplets)

split = int(0.8 * len(triplets))
train_triplets = triplets[:split]
val_triplets   = triplets[split:]

print(f"Train: {len(train_triplets)} triplets")
print(f"Val:   {len(val_triplets)} triplets")

---
## Stage 3 — Format triplets as text prompts

Each triplet is converted to a text sequence using Gemma's native chat template. The loss is computed only on the model's response (the predicted G_t+1), not the prompt — standard causal LM fine-tuning.

```
<start_of_turn>user
Current scene graph:
{G_t as compact JSON}

Action: {action_text}

Predict the next scene graph.<end_of_turn>
<start_of_turn>model
{G_t+1 as compact JSON}<end_of_turn>
```

In [ ]:
def triplet_to_prompt(triplet: dict) -> dict:
    user_msg = (
        f"Current scene graph:\n{json.dumps(triplet['g_t'], separators=(',', ':'))}\n\n"
        f"Action: {triplet['action_text']}\n\n"
        f"Predict the next scene graph."
    )
    target = json.dumps(triplet["g_t1"], separators=(",", ":"))
    return {"input": user_msg, "target": target}


sample = triplet_to_prompt(train_triplets[0])
print("=== INPUT ===")
print(sample["input"])
print("\n=== TARGET ===")
print(sample["target"])

---
## Stage 4 — Load Gemma 3 and attach LoRA

**Why LoRA?** Full fine-tuning of a 4B parameter model requires substantial VRAM for weights and optimizer states. LoRA adds small rank-decomposition matrices to the attention projections and trains only those — approximately 0.5% of total parameters — which fits comfortably on an A100 40GB with 4-bit quantization.

**Hyperparameters:**
- `r=16` — rank of the decomposition. Higher rank increases capacity and memory.
- `lora_alpha=32` — scaling factor, conventionally 2×r.
- Target modules: `q_proj` and `v_proj` (standard for Gemma attention).

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    dtype=torch.bfloat16,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
class TripletDataset(Dataset):
    def __init__(self, triplets, tokenizer, max_length=1024):
        self.samples     = [triplet_to_prompt(t) for t in triplets]
        self.tokenizer   = tokenizer
        self.max_length  = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        messages = [{"role": "user", "content": sample["input"]}]
        prompt = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        full_text = prompt + sample["target"] + self.tokenizer.eos_token

        enc = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"][0]

        # Mask prompt tokens — train only on the target graph tokens
        prompt_ids = self.tokenizer(
            prompt, return_tensors="pt", add_special_tokens=False
        )["input_ids"][0]
        prompt_len = len(prompt_ids)

        labels = input_ids.clone()
        labels[:prompt_len] = -100

        return {"input_ids": input_ids, "attention_mask": enc["attention_mask"][0], "labels": labels}


train_ds = TripletDataset(train_triplets, tokenizer)
val_ds   = TripletDataset(val_triplets,   tokenizer)
print(f"Train dataset: {len(train_ds)} samples")
print(f"Val dataset:   {len(val_ds)} samples")

---
## Stage 5 — Train

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=10,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
)

total_steps = (
    math.ceil(len(train_ds) / training_args.per_device_train_batch_size
              / training_args.gradient_accumulation_steps)
    * training_args.num_train_epochs
)
print(f"Total optimization steps: {total_steps}")
trainer.train()

---
## Stage 6 — Evaluate: graph edit distance

Two metrics are reported:

**Normalized Graph Edit Distance (GED):** The minimum number of node/edge insertions and deletions needed to transform the predicted graph into the ground truth, normalized by the size of the larger graph. Range: [0, 1]. A score of 0 is a perfect prediction.

**Exact match rate:** The fraction of predictions byte-for-byte identical to the ground truth. This is a strict upper bound on quality at pilot scale and is expected to be 0% early on.

In [ ]:
def graph_to_nx(graph: dict) -> nx.Graph:
    G = nx.Graph()
    for obj in graph.get("objects", []):
        state_str = "_".join(f"{k}:{v}" for k, v in obj.get("state", {}).items())
        label = f"{obj['class']}|{state_str}"
        G.add_node(obj["id"], label=label)
    for rel in graph.get("relationships", []):
        G.add_edge(rel["subject"], rel["object"], label=rel["predicate"])
    return G


def compute_ged(g_pred: dict, g_true: dict) -> float:
    """
    Normalized graph edit distance. Returns float in [0, 1].
    Uses networkx with a 2s timeout to bound compute on large graphs.
    """
    G_pred = graph_to_nx(g_pred)
    G_true = graph_to_nx(g_true)
    try:
        ged = nx.graph_edit_distance(G_pred, G_true, timeout=2.0)
        max_size = max(
            G_pred.number_of_nodes() + G_pred.number_of_edges(),
            G_true.number_of_nodes() + G_true.number_of_edges(), 1
        )
        return ged / max_size
    except Exception:
        return 1.0  # timeout treated as maximum distance


def generate_prediction(model, tokenizer, triplet: dict, max_new_tokens=512) -> Optional[dict]:
    prompt   = triplet_to_prompt(triplet)
    messages = [{"role": "user", "content": prompt["input"]}]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    try:
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        return None


print("Evaluation functions defined.")

In [ ]:
model.eval()
results = []

for triplet in tqdm(val_triplets, desc="Evaluating"):
    pred_graph = generate_prediction(model, tokenizer, triplet)

    if pred_graph is None:
        results.append({"action": triplet["action_text"], "ged": 1.0, "exact": False, "parse_ok": False})
        continue

    ged   = compute_ged(pred_graph, triplet["g_t1"])
    exact = (
        json.dumps(pred_graph,      sort_keys=True) ==
        json.dumps(triplet["g_t1"], sort_keys=True)
    )
    results.append({
        "action"      : triplet["action_text"],
        "ged"         : ged,
        "exact"       : exact,
        "parse_ok"    : True,
        "predicted"   : pred_graph,
        "ground_truth": triplet["g_t1"],
    })

parse_rate = sum(r["parse_ok"] for r in results) / len(results)
mean_ged   = sum(r["ged"]      for r in results) / len(results)
exact_rate = sum(r["exact"]    for r in results) / len(results)

print(f"Val set results ({len(results)} examples):")
print(f"  JSON parse rate : {parse_rate:.0%}")
print(f"  Mean norm. GED  : {mean_ged:.3f}  (lower is better, 0 = perfect)")
print(f"  Exact match     : {exact_rate:.0%}")

---
## Stage 7 — Inference demo

Qualitative inspection of per-example predictions against ground truth.

In [ ]:
for r in results:
    print("=" * 60)
    print(f"Action: {r['action']}")
    print(f"GED: {r['ged']:.3f}  |  Exact: {r['exact']}  |  Parse OK: {r['parse_ok']}")
    if r["parse_ok"]:
        print("\nPredicted G_t+1:")
        print(json.dumps(r["predicted"],    indent=2))
        print("\nGround truth G_t+1:")
        print(json.dumps(r["ground_truth"], indent=2))
    print()

---
## Stage 8 — Save checkpoint

Only the LoRA adapter weights are saved (~59 MB vs ~5 GB for the full model). To reload for inference: instantiate the base Gemma 3 model and apply the adapter with `PeftModel.from_pretrained`.

In [ ]:
adapter_path = ROOT / "lora_adapter"
model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))

size_mb = sum(f.stat().st_size for f in adapter_path.rglob("*") if f.is_file()) / 1e6
print(f"Adapter saved to {adapter_path} ({size_mb:.1f} MB)")

In [ ]:
# Save evaluation results for reference in Phase 3
eval_path = ROOT / "eval_results.json"
with open(eval_path, "w") as f:
    json.dump({
        "parse_rate" : parse_rate,
        "mean_ged"   : mean_ged,
        "exact_rate" : exact_rate,
        "per_example": [{k: v for k, v in r.items()} for r in results],
    }, f, indent=2)
print(f"Eval results saved to {eval_path}")

In [ ]:
import shutil
from google.colab import files

shutil.make_archive(str(ROOT / "lora_adapter"), "zip", str(adapter_path))
files.download(str(ROOT / "lora_adapter.zip"))
print("Upload lora_adapter.zip to Phase 3.")